In [ ]:
import subprocess, sys, os, time
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh(f"{sys.executable} -m pip install -q -U transformers peft accelerate bitsandbytes 'trl==0.29.1' 2>&1 | tail -3")
sh(f'{sys.executable} -m pip uninstall -y torchao 2>&1 | tail -1')
sh(f'{sys.executable} -c "import transformers, peft, torch; print(transformers.__version__, peft.__version__, torch.__version__, torch.cuda.device_count(), torch.cuda.get_device_name(0))"')

In [ ]:
import os, glob, json, shutil, hashlib, threading
W = "/kaggle/working"; TMP = "/kaggle/temp"; os.makedirs(TMP, exist_ok=True)
OUT = f"{W}/out"; os.makedirs(OUT, exist_ok=True)
T0 = time.time(); TOM = open(f"{OUT}/tom_tat.txt", "a")
def ghi(*a):
    s = f"[{(time.time()-T0)/3600:5.2f} h] " + " ".join(map(str, a)); print(s, flush=True); TOM.write(s + "\n"); TOM.flush()
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
import torch
assert torch.cuda.device_count() == 2, "DỪNG: cần 2 GPU (T4 x2)"
KIT = os.path.dirname(glob.glob("/kaggle/input/**/gen_aitz.py", recursive=True)[0])
for f in ("grpo_spice.py", "build_branch_data.py", "gen_aitz.py"): shutil.copy(f"{KIT}/{f}", f"{W}/{f}")
os.makedirs(f"{W}/harness", exist_ok=True)
for f in glob.glob(f"{KIT}/harness/*.py"): shutil.copy(f, f"{W}/harness/")
assert md5(f"{W}/grpo_spice.py") == "07ea87b6d156d1faa391a4274dffd7cf", "DỪNG: grpo_spice.py lệch"
AD = f"{KIT}/adapters"
CK = {"ck500": f"{AD}/ck500", "ck500_s202": f"{AD}/ck500_s202"}
assert md5(f"{AD}/adapter_s1_seed101/adapter_model.safetensors") == "0c60a4418d54eaf27bf1d27b895538b7"
assert md5(f"{CK['ck500']}/adapter_model.safetensors") == "491fa6677340393f1e4464c08a0cec98"
assert md5(f"{CK['ck500_s202']}/adapter_model.safetensors") == "57e5d2eec23278b0a2dfe7393d9fc463"
KA = f"{KIT}/kiem_ac"
AZ = [os.path.dirname(t) for t in glob.glob("/kaggle/input/**/aitz_test/test.jsonl", recursive=True)]
assert len(AZ) == 1, f"DỪNG: cần đúng một aitz_test, thấy {AZ}"
AZ = AZ[0]
AZK = os.path.dirname(glob.glob("/kaggle/input/**/only_tap_aitz.jsonl", recursive=True)[0])
assert md5(f"{AZK}/infer_branch.py") == "0eef768c215ce3c3c1fc379e5a46ec32", "DỪNG: infer_branch.py lệch"
assert sum(1 for _ in open(f"{AZK}/only_tap_aitz.jsonl")) == 2736
n_rec = sum(1 for _ in open(f"{AZ}/test.jsonl")); n_ocr = sum(1 for _ in open(f"{AZ}/ocr.jsonl"))
n_img = len(glob.glob(f"{AZ}/images/*.png"))
ghi("AitZ", AZ, "· bản ghi", n_rec, "· OCR", n_ocr, "· ảnh", n_img)
assert n_rec == 4724 and n_ocr == 4724 and n_img == 4724, "DỪNG: AitZ thiếu"
MERGED = f"{TMP}/s1_merged"
def chay(ten, cmd, gpu, cwd=W):
    log = f"{OUT}/{ten}.log"; t = time.time()
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1"}
    with open(log, "a") as f:
        q = subprocess.Popen(cmd, cwd=cwd, stdout=f, stderr=subprocess.STDOUT, env=env, start_new_session=True)
        while q.poll() is None:
            time.sleep(120)
            L = [l.strip() for l in open(log, errors="ignore") if l.strip()]
            print(f"  {(time.time()-T0)/3600:5.2f} h · GPU{gpu} · {ten} · {L[-1][:100] if L else ''}", flush=True)
            if time.time() - T0 > 11.3 * 3600: q.terminate(); ghi("chạm trần giờ —", ten); break
    ghi(ten, "GPU", gpu, "mã thoát", q.returncode, f"· {(time.time()-t)/3600:.2f} h")
    return q.returncode

In [ ]:
r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", AD, "--merged", MERGED], cwd=W, capture_output=True, text=True,
                   env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
print((r.stdout + r.stderr)[-1200:]); assert r.returncode == 0, "DỪNG: hoà lỗi"
# kiểm hoà: S1 hoà trên 20 bước chạm AndroidControl phải trùng preds_s1_seed101 >= 16/20 (luật của các lượt test trước)
kh = f"{OUT}/kiem_hoa_ac.jsonl"
if os.path.exists(kh): os.remove(kh)
chay("kiem_hoa_ac", ["python", "gen_aitz.py", "--bundle", AD, "--merged", MERGED, "--s1", f"{KA}/preds_s1_seed101.jsonl",
     "--recs", f"{KA}/test.jsonl", "--ocr", f"{KA}/ocr.jsonl", "--images", f"{KA}/images", "--tap-only", "--no-q4",
     "--expect-rows", "20", "--expect-taps", "20", "--out", kh], 0)
d = [l for l in open(f"{OUT}/kiem_hoa_ac.log") if "[kiểm hoà]" in l]; ghi(d)
x = int(d[-1].split("]")[1].split("/")[0]); assert x >= 16, f"DỪNG: kiểm hoà {x}/20"

In [ ]:
AZ_ARGS = ["--recs", f"{AZ}/test.jsonl", "--ocr", f"{AZ}/ocr.jsonl", "--images", f"{AZ}/images", "--tap-only", "--no-q4",
           "--expect-rows", "4724", "--expect-taps", "2736"]
def gen(nh):
    if nh == "base":   # mô hình gốc: cùng đường infer_branch đã sinh Base 3B trên AndroidControl
        return ["python", f"{AZK}/infer_branch.py", "--no-adapter", "--data-root", AZ, "--recs-file", "test.jsonl",
                "--only", f"{AZK}/only_tap_aitz.jsonl", "--batch", "4", "--out", f"{OUT}/pred_base_aitz.jsonl"]
    ck = ["--ckpt", CK[nh]] if nh in CK else []
    return ["python", "gen_aitz.py", "--bundle", AD, "--merged", MERGED, *ck, *AZ_ARGS, "--out", f"{OUT}/pred_{nh}_aitz.jsonl"]
def score(nh):
    return ["python", "-u", "harness/score_run.py", "--mode", "score", "--grounder", "uground", "--data-root", AZ,
            "--preds", f"{OUT}/pred_{nh}_aitz.jsonl", "--out", f"{OUT}/score_{nh}_aitz.json"]
# hàng việc: sinh câu trước, chấm khi câu đủ 2.736; hai GPU lấy việc theo thứ tự
NH = ["s1", "ck500", "ck500_s202", "base"]
viec = [("gen", n) for n in NH] + [("score", n) for n in NH]
khoa = threading.Lock(); xong = set()
def du(nh):
    p = f"{OUT}/pred_{nh}_aitz.jsonl"; return os.path.exists(p) and sum(1 for _ in open(p)) == 2736
def tho(gpu):
    while True:
        with khoa:
            san = [v for v in viec if v[0] == "gen" or ("gen", v[1]) in xong]
            if not viec: return
            if not san: v = None
            else: v = san[0]; viec.remove(v)
        if v is None: time.sleep(60); continue
        rc = chay(f"{v[0]}_{v[1]}", gen(v[1]) if v[0] == "gen" else score(v[1]), gpu)
        if v[0] == "gen":
            ghi(v[1], "đủ câu" if du(v[1]) else "THIẾU CÂU", sum(1 for _ in open(f"{OUT}/pred_{v[1]}_aitz.jsonl")))
            with khoa: xong.add(v)
        if time.time() - T0 > 11.3 * 3600: return
ts = [threading.Thread(target=tho, args=(g,)) for g in (0, 1)]
for t in ts: t.start()
for t in ts: t.join()
for nh in NH:
    p = f"{OUT}/score_{nh}_aitz_raw.jsonl"
    if os.path.exists(p):
        R = [json.loads(l) for l in open(p)]
        ghi(f"== {nh}: {len(R)} bước · exec {sum(int(r.get('executable', 0)) for r in R)} · {100*sum(int(r.get('executable', 0)) for r in R)/max(len(R),1):.2f}")

In [ ]:
sh(f"cd {W} && zip -qr aitz_gen_out.zip out && ls -la {W}")
ghi("KẾT THÚC")